# Portfolio Optimization (DA6701 Assignment 1)

Long-only mean-variance portfolio on 10 stocks picked from 30 NIFTY constituents (10 sectors, 3 stocks each). Daily returns from Feb 2023 to Feb 2024.

Pipeline:
1. Load daily returns and sector mapping; 67/33 chronological train-test split.
2. Pick 10 stocks by Ward-linkage hierarchical clustering on correlation distance. The representative per cluster is the in-sample Sharpe winner.
3. Estimate annualised mu and Sigma on the train window. Sigma is Ledoit-Wolf shrunk.
4. Solve GMV and Max Sharpe (via frontier sweep) with CVXPY. Long-only, sum-to-1, 25% per-asset cap.
5. Draw the Monte Carlo efficient frontier (1M Dirichlet-sampled portfolios, cap-aware) and overlay the QP solutions plus the CML.
6. Walk-forward backtest with a 252-day rolling lookback and monthly rebalance.

Risk-free rate 6.5% per year. 252 trading days per year.

## Setup and configuration

Imports and hyperparameters. SciPy handles the clustering, scikit-learn provides Ledoit-Wolf, CVXPY (with OSQP) solves the quadratic programs. Every knob (risk-free rate, train fraction, Monte Carlo count, weight cap, rolling-backtest cadence) sits in the CONFIG block below.

In [ ]:
# ============================================================
# DA6701 Assignment 1 — Portfolio Optimization (VS Code, FULL)
# - 10-stock selection via Hierarchical Correlation Tree (HRT/HRP-style)
#   -> clusters from correlation distance, pick best TRAIN Sharpe per cluster
# - Correlation heatmap (selected 10, train)
# - MPT inputs: Ledoit–Wolf covariance (optional) + sample mean returns
# - GMV + MaxSharpe (long-only, sum=1, optional weight cap) via CVXPY (QP)
# - Efficient Frontier via Monte Carlo simulation (>=10,000; default 1,000,000)
# - Weight attribution table (GMV vs MaxSharpe)
# - Walk-forward backtest on TEST window (rolling 252-day lookback, monthly rebalance)
#
# Install:
#   pip install numpy pandas matplotlib scipy scikit-learn openpyxl cvxpy
# ============================================================

from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -------------------------
# 0) CONFIG
# -------------------------
DATA_DIR = Path(r"./")  # <-- change this to your folder
RETURNS_PATH = DATA_DIR / "daily_returns.xlsx"
CATS_PATH    = DATA_DIR / "stock_categories.xlsx"

N_SELECT = 10
RF_ANNUAL = 0.065
TRADING_DAYS = 252
TRAIN_FRAC = 0.67

# Frontier (required >= 10,000)
N_PORTFOLIOS = 1_000_000
CHUNK_SIZE = 200_000
SEED = 42

# Robustness / constraints
USE_LEDOIT_WOLF = True
MAX_WEIGHT = 0.25          # set None to disable per-asset cap
STRICT_DROPNA = True       # True: drop rows with any NaN; False: fill NaN with 0

# Monte Carlo sampling behavior
DIRICHLET_ALPHA = 2.5      # >1 encourages diversified weights (better acceptance with cap)

# Frontier plot
SCATTER_PLOT_POINTS = 120_000
ENVELOPE_BINS = 250
VOL_Q_LOW, VOL_Q_HIGH = 0.002, 0.998

# Selection / visuals
PLOT_DENDROGRAM = False

# QP sweep for MaxSharpe (via frontier targets)
N_TARGETS = 60

# Walk-forward backtest settings
LOOKBACK_WINDOW = 252       # estimation window (1 year)
REBALANCE_PERIOD = 21       # rebalance frequency (~monthly)
WF_TARGETS = 25             # fewer targets for speed each rebalance

# -------------------------
# 1) DEPENDENCIES
# -------------------------
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from sklearn.covariance import LedoitWolf
import cvxpy as cp



## Data loaders and metrics

Excel loaders that detect the date column, rescale percent returns to decimals when needed, and merge sector labels. Helpers compute annualised return (geometric CAGR), volatility, daily Sharpe, max drawdown, and Calmar.

In [ ]:
# -------------------------
# 2) LOADERS
# -------------------------
def load_returns_xlsx(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path)

    # detect date column
    date_col = None
    for c in df.columns:
        if isinstance(c, str) and "date" in c.lower():
            date_col = c
            break

    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.dropna(subset=[date_col]).set_index(date_col)

    # numeric columns only
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    ret = df[num_cols].copy()
    ret = ret.replace([np.inf, -np.inf], np.nan).dropna(how="all")

    # if values look like percentages, scale down
    q99 = ret.abs().quantile(0.99).max()
    if q99 is not None and q99 > 1.5:
        ret = ret / 100.0

    try:
        ret = ret.sort_index()
    except Exception:
        pass

    return ret


def load_categories_xlsx(path: Path, tickers: list[str]) -> pd.DataFrame:
    cats = pd.read_excel(path)
    cats.columns = [str(c).strip() for c in cats.columns]

    best_col = None
    best_match = -1
    ticker_set = set(map(str, tickers))

    for c in cats.columns:
        col_vals = set(cats[c].astype(str).str.strip().tolist())
        match = len(col_vals.intersection(ticker_set))
        if match > best_match:
            best_match = match
            best_col = c

    sector_col = None
    for c in cats.columns:
        if isinstance(c, str) and any(k in c.lower() for k in ["sector", "category", "industry"]):
            sector_col = c
            break

    if best_col is None or best_match <= 0:
        return cats

    out = cats[[best_col] + ([sector_col] if sector_col else [])].copy()
    out = out.rename(columns={best_col: "Ticker"})
    if sector_col:
        out = out.rename(columns={sector_col: "Sector"})
    out["Ticker"] = out["Ticker"].astype(str).str.strip()
    return out


# -------------------------
# 3) UTILS
# -------------------------
def rf_daily_from_annual(rf_annual: float, trading_days: int = 252) -> float:
    return (1.0 + rf_annual) ** (1.0 / trading_days) - 1.0


def prepare_returns_matrix(R: pd.DataFrame) -> pd.DataFrame:
    if STRICT_DROPNA:
        return R.dropna(how="any")
    return R.fillna(0.0)


def max_drawdown(returns: pd.Series) -> float:
    wealth = (1.0 + returns.fillna(0.0)).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1.0
    return float(dd.min())


def perf_metrics_daily_sharpe(returns: pd.Series, rf_annual: float = 0.065, trading_days: int = 252) -> dict:
    """
    Sharpe computed from DAILY returns:
      Sharpe = ((mean_d - rf_d) / std_d) * sqrt(252)
    Annual return is geometric CAGR on the sample length.
    """
    r = returns.dropna()
    if len(r) == 0:
        return {"AnnReturn": np.nan, "AnnVol": np.nan, "Sharpe": np.nan, "MaxDrawdown": np.nan, "Calmar": np.nan}

    rf_d = rf_daily_from_annual(rf_annual, trading_days)
    mean_d = float(r.mean())
    vol_d = float(r.std(ddof=1))
    ann_vol = vol_d * np.sqrt(trading_days)
    sharpe = ((mean_d - rf_d) / vol_d) * np.sqrt(trading_days) if vol_d > 0 else np.nan

    total = float((1.0 + r).prod())
    ann_ret = total ** (trading_days / len(r)) - 1.0

    mdd = max_drawdown(r)
    calmar = ann_ret / abs(mdd) if mdd < 0 else np.nan

    return {
        "AnnReturn": float(ann_ret),
        "AnnVol": float(ann_vol),
        "Sharpe": float(sharpe),
        "MaxDrawdown": float(mdd),
        "Calmar": float(calmar),
    }



## Stock selection: hierarchical correlation tree

Build the correlation matrix on the train split. Convert to a distance `d_ij = sqrt((1 - rho_ij) / 2)`. Run Ward-linkage agglomerative clustering, cut into 10 clusters, and pick the highest in-sample Sharpe member of each cluster (medoid breaks ties). Doing this on the train split only avoids look-ahead bias.

In [ ]:
# -------------------------
# 4) HIERARCHICAL SELECTION (correlation tree + best Sharpe rep)
# -------------------------
def select_10_hierarchical_best_sharpe(
    returns_train: pd.DataFrame,
    k: int = 10,
    linkage_method: str = "ward",
    use_abs_corr: bool = False,
    plot_dendrogram: bool = False,
    rf_annual: float = 0.065
) -> list[str]:
    R = returns_train.dropna(how="any")
    tickers = list(R.columns)
    n_assets = len(tickers)
    if k > n_assets:
        raise ValueError(f"k={k} > n_assets={n_assets}")

    corr = R.corr()
    if use_abs_corr:
        corr = corr.abs()
    corr = corr.clip(-1.0, 1.0)
    np.fill_diagonal(corr.values, 1.0)

    dist = np.sqrt(0.5 * (1.0 - corr))
    np.fill_diagonal(dist.values, 0.0)

    Z = linkage(squareform(dist.values, checks=False), method=linkage_method)

    if plot_dendrogram:
        plt.figure(figsize=(12, 5))
        dendrogram(Z, labels=tickers, leaf_rotation=90)
        plt.title("Hierarchical Clustering Dendrogram (Train Correlation Tree)")
        plt.tight_layout()
        plt.show()

    labels = fcluster(Z, t=k, criterion="maxclust")
    clusters: dict[int, list[str]] = {}
    for tkr, lab in zip(tickers, labels):
        clusters.setdefault(int(lab), []).append(tkr)

    # robustness: if fewer than k clusters, loosen cut
    k_try = k
    while len(clusters) < k and k_try < n_assets:
        k_try += 1
        labels = fcluster(Z, t=k_try, criterion="maxclust")
        clusters = {}
        for tkr, lab in zip(tickers, labels):
            clusters.setdefault(int(lab), []).append(tkr)

    rf_d = rf_daily_from_annual(rf_annual, TRADING_DAYS)
    mu = R.mean()
    vol = R.std(ddof=1).replace(0, np.nan)
    sharpe = (mu - rf_d) / vol

    print("\nHierarchical selection (best Sharpe representative per cluster):")
    print(f"  linkage={linkage_method}, use_abs_corr={use_abs_corr}, clusters_found={len(clusters)} (target={k})")

    selected: list[str] = []
    cluster_items = sorted(clusters.items(), key=lambda kv: len(kv[1]), reverse=True)

    for i, (_, members) in enumerate(cluster_items, start=1):
        members = list(members)
        sh = sharpe.loc[members].replace([np.inf, -np.inf], np.nan).fillna(-1e9)
        best_sh = float(sh.max())
        best_members = sh[sh == best_sh].index.tolist()

        if len(best_members) == 1:
            rep = best_members[0]
        else:
            # tie-break among best by medoid
            D = dist.loc[best_members, best_members].values
            avg_d = D.mean(axis=1)
            rep = best_members[int(np.argmin(avg_d))]

        selected.append(rep)
        print(f"  Cluster {i}: size={len(members):2d} | pick={rep} | best Sharpe (daily)={best_sh:.4f}")

    if len(selected) > k:
        selected = selected[:k]

    # fallback fill if needed
    if len(selected) < k:
        remaining = [t for t in tickers if t not in selected]
        corr_abs = R.corr().abs()
        while len(selected) < k and remaining:
            scores = {cand: float(corr_abs.loc[cand, selected].mean()) for cand in remaining}
            nxt = min(scores, key=scores.get)
            selected.append(nxt)
            remaining.remove(nxt)
            print(f"  Fallback add: {nxt} (mean |corr| to selected={scores[nxt]:.4f})")

    return selected



## Estimation and convex optimisation

Annualised mu and Sigma from the train window. Sigma is shrunk with Ledoit-Wolf to stabilise the 10x10 matrix on ~500 daily observations. Two convex programs, both solved with CVXPY (OSQP):

- GMV (Global Minimum Variance): minimise `w' Sigma w` subject to `sum(w) = 1`, `0 <= w <= cap`.
- Max Sharpe via frontier sweep: solve `min w' Sigma w s.t. mu' w >= target` for 60 target returns, then keep the weights with the highest realised Sharpe. Sidesteps the non-convex direct Sharpe maximisation.

In [ ]:
# -------------------------
# 5) MU/COV (annualized)
# -------------------------
def mu_cov_annual(R_daily: pd.DataFrame, use_ledoit_wolf: bool = True) -> tuple[np.ndarray, np.ndarray]:
    R = prepare_returns_matrix(R_daily)
    mu_daily = R.mean().values.astype(np.float64)

    if use_ledoit_wolf:
        lw = LedoitWolf().fit(R.values)
        cov_daily = lw.covariance_.astype(np.float64)
        print(f"Ledoit–Wolf shrinkage: {float(lw.shrinkage_):.6f}")
    else:
        cov_daily = R.cov().values.astype(np.float64)

    mu_ann = mu_daily * TRADING_DAYS
    cov_ann = cov_daily * TRADING_DAYS
    return mu_ann, cov_ann


# -------------------------
# 6) QP OPTIMIZERS
# -------------------------
def solve_gmv(cov_ann: np.ndarray, cap: float | None = None) -> np.ndarray:
    n = cov_ann.shape[0]
    w = cp.Variable(n)
    obj = cp.Minimize(cp.quad_form(w, cov_ann))
    cons = [cp.sum(w) == 1, w >= 0]
    if cap is not None:
        cons.append(w <= cap)
    prob = cp.Problem(obj, cons)
    prob.solve(solver=cp.OSQP, verbose=False)

    if w.value is None:
        raise RuntimeError("GMV optimization failed.")
    ww = np.array(w.value).reshape(-1)
    ww = np.clip(ww, 0, None)
    ww = ww / ww.sum()
    return ww


def solve_minvar_for_target(mu_ann: np.ndarray, cov_ann: np.ndarray, target_ret: float,
                            cap: float | None = None) -> np.ndarray | None:
    n = len(mu_ann)
    w = cp.Variable(n)
    obj = cp.Minimize(cp.quad_form(w, cov_ann))
    cons = [cp.sum(w) == 1, w >= 0, mu_ann @ w >= target_ret]
    if cap is not None:
        cons.append(w <= cap)
    prob = cp.Problem(obj, cons)
    prob.solve(solver=cp.OSQP, verbose=False)

    if w.value is None:
        return None
    ww = np.array(w.value).reshape(-1)
    ww = np.clip(ww, 0, None)
    if ww.sum() <= 0:
        return None
    ww = ww / ww.sum()
    return ww


def portfolio_stats(mu_ann: np.ndarray, cov_ann: np.ndarray, w: np.ndarray, rf_ann: float) -> tuple[float, float, float]:
    ret = float(mu_ann @ w)
    vol = float(np.sqrt(max(w @ cov_ann @ w, 1e-18)))
    sharpe = (ret - rf_ann) / max(vol, 1e-12)
    return ret, vol, sharpe


def solve_maxsharpe_via_frontier(mu_ann: np.ndarray, cov_ann: np.ndarray, rf_ann: float,
                                 cap: float | None = None, n_targets: int = 60) -> np.ndarray:
    mu_min = float(np.min(mu_ann))
    mu_max = float(np.max(mu_ann))

    lo = mu_min + 0.10 * (mu_max - mu_min)
    hi = mu_min + 0.95 * (mu_max - mu_min)
    targets = np.linspace(lo, hi, n_targets)

    best_w = None
    best_sh = -np.inf

    for t in targets:
        w = solve_minvar_for_target(mu_ann, cov_ann, target_ret=float(t), cap=cap)
        if w is None:
            continue
        _, _, sh = portfolio_stats(mu_ann, cov_ann, w, rf_ann)
        if sh > best_sh:
            best_sh = sh
            best_w = w

    if best_w is None:
        return solve_gmv(cov_ann, cap=cap)
    return best_w



## Monte Carlo efficient frontier

One million long-only weight vectors sampled from a Dirichlet distribution. Rejection sampling enforces the per-asset cap. Returns and volatilities are vectorised in float32 chunks of 200,000 to bound memory. An upper envelope of the cloud is built by bucketing volatilities and taking the highest return in each bucket.

In [ ]:
# -------------------------
# 7) MONTE CARLO FRONTIER (required plot)
# -------------------------
def sample_weights_capped(rng: np.random.Generator, n_needed: int, n_assets: int,
                          cap: float | None, alpha: float) -> np.ndarray:
    if cap is None:
        return rng.dirichlet(alpha=np.full(n_assets, alpha), size=n_needed).astype(np.float32)

    cap = float(cap)
    if n_assets * cap < 1.0 - 1e-12:
        raise ValueError(f"Infeasible cap={cap} for n_assets={n_assets} because n*cap < 1.")

    out = np.empty((n_needed, n_assets), dtype=np.float32)
    filled = 0
    oversample = 3

    while filled < n_needed:
        m = n_needed - filled
        m_gen = max(2000, oversample * m)
        W = rng.dirichlet(alpha=np.full(n_assets, alpha), size=m_gen).astype(np.float32)
        ok = (W.max(axis=1) <= cap + 1e-9)
        W_ok = W[ok]
        take = min(len(W_ok), m)
        if take > 0:
            out[filled:filled+take] = W_ok[:take]
            filled += take
    return out


def run_monte_carlo(mu_ann, cov_ann, rf=0.065, n_port=1_000_000, chunk=200_000, seed=42,
                    cap: float | None = None, alpha: float = 2.5):
    rng = np.random.default_rng(seed)
    n_assets = len(mu_ann)

    rets = np.empty(n_port, dtype=np.float32)
    vols = np.empty(n_port, dtype=np.float32)
    sharpes = np.empty(n_port, dtype=np.float32)

    idx = 0
    mu = mu_ann.astype(np.float64)
    cov = cov_ann.astype(np.float64)

    while idx < n_port:
        m = min(chunk, n_port - idx)
        W = sample_weights_capped(rng, m, n_assets, cap=cap, alpha=alpha)

        pret = (W @ mu).astype(np.float32)
        pvar = np.einsum("ij,jk,ik->i", W.astype(np.float64), cov, W.astype(np.float64))
        pvol = np.sqrt(np.maximum(pvar, 1e-18)).astype(np.float32)
        psh = ((pret - rf) / np.maximum(pvol, 1e-12)).astype(np.float32)

        rets[idx:idx+m] = pret
        vols[idx:idx+m] = pvol
        sharpes[idx:idx+m] = psh
        idx += m

    return {"rets": rets, "vols": vols, "sharpes": sharpes}


def upper_envelope(vols, rets, bins=250, qlow=0.002, qhigh=0.998):
    v = vols.astype(np.float64)
    r = rets.astype(np.float64)

    vmin = np.quantile(v, qlow)
    vmax = np.quantile(v, qhigh)

    mask = (v >= vmin) & (v <= vmax) & np.isfinite(v) & np.isfinite(r)
    v = v[mask]; r = r[mask]

    edges = np.linspace(v.min(), v.max(), bins + 1)
    idx = np.clip(np.digitize(v, edges) - 1, 0, bins - 1)

    best_r = np.full(bins, -np.inf)
    best_v = np.full(bins, np.nan)

    for i in range(bins):
        m = (idx == i)
        if np.any(m):
            j = np.argmax(r[m])
            best_r[i] = r[m][j]
            best_v[i] = v[m][j]

    ok = np.isfinite(best_v) & np.isfinite(best_r) & (best_r > -np.inf)
    best_v = best_v[ok]
    best_r = best_r[ok]

    order = np.argsort(best_v)
    best_v = best_v[order]
    best_r = best_r[order]
    best_r = np.maximum.accumulate(best_r)

    return best_v, best_r



## Walk-forward backtest

Out-of-sample test on the final third of the data. Every 21 trading days, re-fit mu and Sigma on the previous 252 days, resolve both QPs, and hold those weights until the next rebalance. Records a daily return series for GMV, Max Sharpe and equal-weight, plus the full weights history.

In [ ]:
# -------------------------
# 8) WALK-FORWARD BACKTEST (TEST window)
# -------------------------
def mu_cov_annual_window(R_window: pd.DataFrame, use_ledoit_wolf: bool = True) -> tuple[np.ndarray, np.ndarray]:
    Rw = prepare_returns_matrix(R_window)
    if len(Rw) < 30:
        raise ValueError(f"Too few rows in window after NA handling: {len(Rw)}")

    mu_daily = Rw.mean().values.astype(np.float64)

    if use_ledoit_wolf:
        lw = LedoitWolf().fit(Rw.values)
        cov_daily = lw.covariance_.astype(np.float64)
    else:
        cov_daily = Rw.cov().values.astype(np.float64)

    mu_ann = mu_daily * TRADING_DAYS
    cov_ann = cov_daily * TRADING_DAYS
    return mu_ann, cov_ann


def walk_forward_backtest(
    returns_all_selected: pd.DataFrame,
    test_start: pd.Timestamp,
    test_end: pd.Timestamp,
    lookback_window: int = 252,
    rebalance_period: int = 21,
    rf_annual: float = 0.065,
    cap: float | None = 0.25,
    use_ledoit_wolf: bool = True,
    n_targets: int = 25
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Rolling (no-lookahead):
      - At each rebalance date in test window:
          estimate mu/cov from previous lookback_window days
          compute GMV + MaxSharpe weights
          apply to next holding period returns
    Returns:
      daily_pf (GMV, MaxSharpe, EqualWeight daily returns),
      weights_hist (weights at each rebalance date)
    """
    R = returns_all_selected.sort_index()

    test_start = pd.to_datetime(test_start)
    test_end = pd.to_datetime(test_end)

    # snap to nearest index if exact dates aren't present
    if test_start not in R.index:
        test_start = R.index[R.index.get_indexer([test_start], method="nearest")[0]]
    if test_end not in R.index:
        test_end = R.index[R.index.get_indexer([test_end], method="nearest")[0]]

    start_loc = R.index.get_loc(test_start)
    end_loc = R.index.get_loc(test_end)

    if start_loc - lookback_window < 0:
        raise ValueError("Not enough history before test_start for lookback_window.")

    tickers = list(R.columns)
    n_assets = len(tickers)
    w_eq = np.ones(n_assets, dtype=np.float64) / n_assets

    eval_index = R.index[start_loc:end_loc+1]
    out = pd.DataFrame(index=eval_index, columns=["GMV", "MaxSharpe", "EqualWeight"], dtype=float)

    rebalance_locs = list(range(start_loc, end_loc + 1, rebalance_period))
    weights_rows = []

    for i, reb_loc in enumerate(rebalance_locs):
        # estimate window: [reb_loc - lookback_window, reb_loc)
        est_start = reb_loc - lookback_window
        est_end = reb_loc
        R_win = R.iloc[est_start:est_end]

        try:
            mu_ann, cov_ann = mu_cov_annual_window(R_win, use_ledoit_wolf=use_ledoit_wolf)
            w_gmv = solve_gmv(cov_ann, cap=cap)
            w_ms = solve_maxsharpe_via_frontier(mu_ann, cov_ann, rf_ann=rf_annual, cap=cap, n_targets=n_targets)
        except Exception as e:
            print(f"[WARN] Rebalance at {R.index[reb_loc].date()} skipped: {e}")
            continue

        # holding period: [reb_loc, next_reb_loc)
        hold_start = reb_loc
        hold_end = rebalance_locs[i+1] if (i+1) < len(rebalance_locs) else (end_loc + 1)
        R_hold = prepare_returns_matrix(R.iloc[hold_start:hold_end])

        if len(R_hold) == 0:
            continue

        out.loc[R_hold.index, "GMV"] = R_hold.values @ w_gmv
        out.loc[R_hold.index, "MaxSharpe"] = R_hold.values @ w_ms
        out.loc[R_hold.index, "EqualWeight"] = R_hold.values @ w_eq

        row = {"RebalanceDate": R.index[reb_loc]}
        for t, w in zip(tickers, w_gmv):
            row[f"GMV_{t}"] = float(w)
        for t, w in zip(tickers, w_ms):
            row[f"MS_{t}"] = float(w)
        weights_rows.append(row)

    weights_hist = pd.DataFrame(weights_rows).set_index("RebalanceDate").sort_index()
    out = out.dropna(how="all")

    return out, weights_hist



## Run

In [ ]:
# -------------------------
# 9) MAIN
# -------------------------
def main():
    # Load data
    returns_all = load_returns_xlsx(RETURNS_PATH)
    tickers_all = list(returns_all.columns)
    cats_df = load_categories_xlsx(CATS_PATH, tickers_all)

    print("Returns shape:", returns_all.shape)
    print("First 5 tickers:", tickers_all[:5])
    print("\nCategories preview:")
    print(cats_df.head().to_string(index=False))

    # Split train/test
    split = int(TRAIN_FRAC * len(returns_all))
    returns_train_full = returns_all.iloc[:split].copy()
    returns_test_full  = returns_all.iloc[split:].copy()

    print("\nTrain rows:", len(returns_train_full), "Test rows:", len(returns_test_full))

    # Select 10 (train only)
    selected10 = select_10_hierarchical_best_sharpe(
        returns_train_full,
        k=N_SELECT,
        linkage_method="ward",
        use_abs_corr=False,
        plot_dendrogram=PLOT_DENDROGRAM,
        rf_annual=RF_ANNUAL
    )
    print("\nSelected 10:", selected10)

    # Show selected + sector
    selected_info = pd.DataFrame({"Ticker": selected10})
    if "Ticker" in cats_df.columns:
        selected_info = selected_info.merge(cats_df, on="Ticker", how="left")
    print("\nSelected tickers + sector info:")
    print(selected_info.to_string(index=False))

    # Correlation heatmap on train
    corr10 = returns_train_full[selected10].corr()
    plt.figure(figsize=(9, 7))
    plt.imshow(corr10.values, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(N_SELECT), selected10, rotation=90)
    plt.yticks(range(N_SELECT), selected10)
    plt.title("Correlation Heatmap (Selected 10, Train)")
    plt.tight_layout()
    plt.show()

    upper = corr10.where(np.triu(np.ones_like(corr10, dtype=bool), k=1))
    avg_abs_corr_10 = upper.abs().stack().mean()
    print(f"\nAvg |corr| among selected 10 (train): {avg_abs_corr_10:.4f}")

    # MPT inputs on train (annualized)
    R_train = prepare_returns_matrix(returns_train_full[selected10])
    print(f"\nTrain usable rows after NA handling: {len(R_train)}")
    mu_ann, cov_ann = mu_cov_annual(R_train, use_ledoit_wolf=USE_LEDOIT_WOLF)

    # QP solutions (GMV + MaxSharpe)
    w_gmv = solve_gmv(cov_ann, cap=MAX_WEIGHT)
    w_ms  = solve_maxsharpe_via_frontier(mu_ann, cov_ann, rf_ann=RF_ANNUAL, cap=MAX_WEIGHT, n_targets=N_TARGETS)

    gmv_ret, gmv_vol, gmv_sh = portfolio_stats(mu_ann, cov_ann, w_gmv, RF_ANNUAL)
    ms_ret,  ms_vol,  ms_sh  = portfolio_stats(mu_ann, cov_ann, w_ms,  RF_ANNUAL)
    print("\nQP GMV (train est.):", {"ret": gmv_ret, "vol": gmv_vol, "sharpe": gmv_sh})
    print("QP MaxSharpe (train est.):", {"ret": ms_ret, "vol": ms_vol, "sharpe": ms_sh})

    # Monte Carlo frontier (required)
    mc = run_monte_carlo(
        mu_ann, cov_ann, rf=RF_ANNUAL,
        n_port=N_PORTFOLIOS, chunk=CHUNK_SIZE, seed=SEED,
        cap=MAX_WEIGHT, alpha=DIRICHLET_ALPHA
    )

    vols = mc["vols"]
    rets = mc["rets"]

    rng = np.random.default_rng(SEED)
    plot_n = min(SCATTER_PLOT_POINTS, len(vols))
    plot_idx = rng.choice(len(vols), size=plot_n, replace=False)

    env_v, env_r = upper_envelope(vols, rets, bins=ENVELOPE_BINS, qlow=VOL_Q_LOW, qhigh=VOL_Q_HIGH)

    plt.figure(figsize=(10, 6))
    plt.scatter(vols[plot_idx], rets[plot_idx], s=6, alpha=0.25)
    plt.plot(env_v, env_r, linewidth=2)

    plt.scatter([gmv_vol], [gmv_ret], marker="*", s=280)
    plt.scatter([ms_vol], [ms_ret], marker="*", s=280)

    # CML from QP MaxSharpe
    cml_slope = ms_sh
    cml_x = np.linspace(0.0, max(env_v.max(), ms_vol) * 1.05, 200)
    cml_y = RF_ANNUAL + cml_slope * cml_x
    plt.plot(cml_x, cml_y, linewidth=2)

    plt.xlabel("Annualized Volatility")
    plt.ylabel("Annualized Return")
    plt.title(f"Monte Carlo Efficient Frontier ({N_PORTFOLIOS:,} portfolios)\nStars = QP GMV & QP Max Sharpe, Line = CML")
    plt.tight_layout()
    plt.show()

    # Weights table (deliverable)
    weights_df = pd.DataFrame({
        "Ticker": selected10,
        "GMV_weight": np.round(w_gmv, 6),
        "MaxSharpe_weight": np.round(w_ms, 6),
    })
    print("\nWeights (GMV vs MaxSharpe):")
    print(weights_df.to_string(index=False))

    # ---------- Walk-forward backtest on TEST window ----------
    # Use full history for selected10 so test start can look back into train
    R_full_sel = returns_all[selected10].copy()
    test_start = returns_test_full.index[0]
    test_end   = returns_test_full.index[-1]

    daily_pf, weights_hist = walk_forward_backtest(
        returns_all_selected=R_full_sel,
        test_start=test_start,
        test_end=test_end,
        lookback_window=LOOKBACK_WINDOW,
        rebalance_period=REBALANCE_PERIOD,
        rf_annual=RF_ANNUAL,
        cap=MAX_WEIGHT,
        use_ledoit_wolf=USE_LEDOIT_WOLF,
        n_targets=WF_TARGETS
    )

    print("\nWalk-forward daily returns shape:", daily_pf.shape)
    if not weights_hist.empty:
        print("\nWeights history (first 3 rebalances):")
        print(weights_hist.head(3).to_string())

    summary = pd.DataFrame([
        {"Portfolio": c, **perf_metrics_daily_sharpe(daily_pf[c], rf_annual=RF_ANNUAL)}
        for c in daily_pf.columns
    ])
    print("\nWALK-FORWARD (TEST) performance summary (daily Sharpe):")
    print(summary.to_string(index=False))

    # Equity curve plot
    equity = (1.0 + daily_pf.fillna(0.0)).cumprod()
    plt.figure(figsize=(10, 5))
    for col in equity.columns:
        plt.plot(equity.index, equity[col], label=col)
    plt.title("Walk-forward Equity Curves (Test Window)")
    plt.legend()
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()
